# 🎹 Aria — Chopin Style Finetuning
**Goal:** Finetune EleutherAI Aria on Chopin MIDI files so it generates music with Chopin's specific harmonic language, ornamental style, and emotional depth.

**Method:** LoRA (Low-Rank Adaptation) — efficient finetuning that modifies a small number of parameters, preserving the model's piano/musical knowledge while shifting its style toward Chopin.

### What you need
- A folder of **Chopin MIDI files** (80–200 files is enough). Performance MIDIs are better than score MIDIs.
- **Colab A100 GPU** (recommended) or T4 (slower but works)
- This will take **1–3 hours** for a good finetune on T4

### Where to get Chopin MIDIs (free)
- [Kunstderfuge](http://www.kunstderfuge.com/chopin.htm) — large collection of performance MIDIs
- [Mutopia Project](https://www.mutopiaproject.org/cgibin/make-table.cgi?Composer=ChopinFF) — free, clean scores
- [Piano MIDI de](http://www.piano-midi.de/chopin.htm) — high quality performance MIDIs
- [IMSLP](https://imslp.org/) — search "Chopin" → MIDI files section

**Tip:** Mix different genres (nocturnes, mazurkas, waltzes, etudes, preludes) for a more general Chopin style rather than one that only sounds like one piece type.

---

**Runtime:** Use **A100** (Colab Pro) if available. T4 will work but is ~3x slower.

## Step 1 — Install Dependencies

In [1]:
!pip install -q git+https://github.com/EleutherAI/aria-utils.git
!pip install -q transformers safetensors huggingface_hub peft accelerate
!pip install -q pretty_midi tqdm
!pip install -q "torchao>=0.16.0"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Dependencies installed")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 44.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.8 MB/s eta 0:00:00

✅ Dependencies installed
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Step 2 — Upload Your Chopin MIDI Dataset

Upload a zip file containing all your Chopin MIDI files, or upload them individually.

In [2]:
import os, zipfile, glob
from google.colab import files

MIDI_DIR = "chopin_midi"
os.makedirs(MIDI_DIR, exist_ok=True)

print("Upload your Chopin MIDI files (zip or individual .mid files):")
print("→ If you have a zip, upload that. If individual files, upload them all at once.")
print()

uploaded = files.upload()

for fname, data in uploaded.items():
    fpath = os.path.join(MIDI_DIR, fname)
    with open(fpath, 'wb') as f:
        f.write(data)

    # Unzip if it's a zip
    if fname.endswith('.zip'):
        print(f"Extracting {fname}...")
        with zipfile.ZipFile(fpath, 'r') as zf:
            zf.extractall(MIDI_DIR)
        os.remove(fpath)

# Collect all MIDI files recursively
midi_files = glob.glob(f"{MIDI_DIR}/**/*.mid", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/**/*.midi", recursive=True) + \
             glob.glob(f"{MIDI_DIR}/*.mid") + \
             glob.glob(f"{MIDI_DIR}/*.midi")
midi_files = sorted(set(midi_files))

print(f"\n✅ Found {len(midi_files)} MIDI files")
if len(midi_files) < 20:
    print("⚠️  Less than 20 files — the model can still finetune but style transfer will be limited.")
    print("   Recommended: 80+ files for a recognizable Chopin style.")
elif len(midi_files) >= 80:
    print("👍 Great dataset size! This should produce a strong Chopin style.")

Upload your Chopin MIDI files (zip or individual .mid files):
→ If you have a zip, upload that. If individual files, upload them all at once.



Saving Chopin-PianoOnly.zip to Chopin-PianoOnly.zip
Extracting Chopin-PianoOnly.zip...

✅ Found 188 MIDI files
👍 Great dataset size! This should produce a strong Chopin style.


## Step 3 — Load the Base Model + Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import transformers.tokenization_utils as _tok_utils

print("Loading Aria model (downloading ~2.8 GB on first run)...")

# ── Fix 1: patch missing initializer_range ──
config = AutoConfig.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)
if not hasattr(config, "initializer_range"):
    config.initializer_range = 0.02

model = AutoModelForCausalLM.from_pretrained(
    "loubb/aria-medium-base",
    config=config,
    trust_remote_code=True,
    dtype=torch.float16,
)

# ── Load gen checkpoint as starting point ──
print("Loading generation checkpoint as starting point...")
gen_ckpt_path = hf_hub_download(
    repo_id="loubb/aria-medium-base",
    filename="model-gen.safetensors"
)
gen_state_dict = load_file(gen_ckpt_path)
model.load_state_dict(gen_state_dict, strict=False)

# ── Fix 2: inject BatchEncoding into tokenization_utils namespace ──
if not hasattr(_tok_utils, "BatchEncoding"):
    from transformers.tokenization_utils_base import BatchEncoding
    _tok_utils.BatchEncoding = BatchEncoding
    print("Patched BatchEncoding into tokenization_utils namespace")

tokenizer = AutoTokenizer.from_pretrained(
    "loubb/aria-medium-base",
    trust_remote_code=True,
)

# ── Fix 3: patch unimplemented save_vocabulary ──
tokenizer.save_vocabulary = lambda save_directory, filename_prefix=None: ()

print("✅ Base model loaded")

## Step 4 — Apply LoRA for Efficient Finetuning

In [4]:
from peft import LoraConfig, get_peft_model, TaskType

LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05

# Aria's actual layer names (confirmed from inspection)
TARGET_MODULES = [
    "att_proj_linear",  # combined QKV + output attention projection
    "ff_gate_proj",     # feedforward gate
    "ff_up_proj",       # feedforward up
    "ff_down_proj",     # feedforward down
]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model = model.to(DEVICE)
print("\n✅ LoRA applied — model is ready to finetune")

trainable params: 13,369,344 || all params: 671,907,840 || trainable%: 1.9898

✅ LoRA applied — model is ready to finetune


In [5]:
from peft import LoraConfig, get_peft_model, TaskType

# ╔══════════════════════════════════════════╗
# ║           LORA CONFIGURATION            ║
# ╚══════════════════════════════════════════╝

LORA_R          = 32    # LoRA rank. Higher = more capacity but more VRAM
                        #   16 = light finetuning (safe for T4)
                        #   32 = balanced (recommended, works on T4+A100)
                        #   64 = heavy finetuning (A100 recommended)

LORA_ALPHA      = 64    # Usually 2x LORA_R is a good default

LORA_DROPOUT    = 0.05  # Small dropout to avoid overfitting on a small dataset

# Target the attention layers — these control musical style and pattern recognition
# q_proj and v_proj give the best style transfer for music
TARGET_MODULES = [
    "att_proj_linear",  # combined QKV + output attention projection
    "ff_gate_proj",     # feedforward gate
    "ff_up_proj",       # feedforward up
    "ff_down_proj",     # feedforward down
]

# ══════════════════════════════════════════

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Move to device after LoRA wrapping
model = model.to(DEVICE)
print("\n✅ LoRA applied — model is ready to finetune")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 13,369,344 || all params: 671,907,840 || trainable%: 1.9898

✅ LoRA applied — model is ready to finetune


## Step 5 — Prepare the Dataset

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random

SEQ_LEN = 1024
STRIDE  = 512

class ChopinMIDIDataset(Dataset):
    def __init__(self, midi_files, tokenizer, seq_len, stride):
        self.samples = []
        failed = 0

        print(f"Tokenizing {len(midi_files)} MIDI files...")
        for path in tqdm(midi_files):
            try:
                result = tokenizer.encode_from_file(path)

                # Aria tokenizer returns a BatchEncoding — extract input_ids
                if hasattr(result, '__getitem__') and 'input_ids' in result:
                    token_ids = result['input_ids']
                    # input_ids is a list of lists (batch dim) — unwrap it
                    if isinstance(token_ids, list) and isinstance(token_ids[0], list):
                        token_ids = token_ids[0]
                elif isinstance(result, list):
                    token_ids = result
                else:
                    token_ids = result.tolist()

                if len(token_ids) < 64:
                    failed += 1
                    continue

                # Chop into overlapping windows of SEQ_LEN
                for start in range(0, max(1, len(token_ids) - seq_len), stride):
                    chunk = token_ids[start : start + seq_len]
                    if len(chunk) < 64:
                        continue
                    if len(chunk) < seq_len:
                        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id else 0
                        chunk = chunk + [pad_id] * (seq_len - len(chunk))
                    self.samples.append(torch.tensor(chunk, dtype=torch.long))

            except Exception as e:
                failed += 1

        random.shuffle(self.samples)
        print(f"\n✅ Dataset ready: {len(self.samples)} training sequences")
        print(f"   From {len(midi_files) - failed} valid files ({failed} failed to parse)")
        print(f"   Approx. total music: ~{len(self.samples) * seq_len * 0.03 / 60:.0f} min")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return {"input_ids": ids, "labels": ids.clone()}


dataset = ChopinMIDIDataset(midi_files, tokenizer, SEQ_LEN, STRIDE)

if len(dataset) == 0:
    raise ValueError("No training samples were created! Check that your MIDI files are valid.")

Tokenizing 188 MIDI files...


100%|██████████| 188/188 [01:13<00:00,  2.55it/s]


✅ Dataset ready: 2047 training sequences
   From 188 valid files (0 failed to parse)
   Approx. total music: ~1048 min


## Step 6 — Train! 🎼

Adjust training settings below. For a first run, the defaults are safe.

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ╔══════════════════════════════════════════╗
# ║        TRAINING CONFIGURATION           ║
# ╚══════════════════════════════════════════╝

NUM_EPOCHS       = 3
BATCH_SIZE       = 2
GRAD_ACCUM       = 8
LEARNING_RATE    = 3e-5
SAVE_STEPS       = 100
OUTPUT_DIR       = "aria_chopin_lora"  #

# ══════════════════════════════════════════

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=,        # Keep only the 3 most recent checkpoints
    report_to="none",          # Disable wandb/tensorboard
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

# Data collator — pads/batches sequences for causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

# Estimate training time
steps_per_epoch = len(dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"Training plan:")
print(f"  Dataset size    : {len(dataset)} sequences")
print(f"  Steps per epoch : ~{steps_per_epoch}")
print(f"  Total steps     : ~{total_steps}")
print(f"  Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"\n  ⏱ Estimated time on T4  : ~{total_steps * 4 / 60:.0f}–{total_steps * 6 / 60:.0f} min")
print(f"  ⏱ Estimated time on A100: ~{total_steps * 1.5 / 60:.0f}–{total_steps * 2.5 / 60:.0f} min")
print()
print("Starting training...")
print("Watch the loss — it should drop from ~5–7 down to ~2–3 for a good finetune.")
print()

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training plan:
  Dataset size    : 2047 sequences
  Steps per epoch : ~127
  Total steps     : ~381
  Effective batch : 16

  ⏱ Estimated time on T4  : ~25–38 min
  ⏱ Estimated time on A100: ~10–16 min

Starting training...
Watch the loss — it should drop from ~5–7 down to ~2–3 for a good finetune.



TypeError: AriaForCausalLM.forward() got an unexpected keyword argument 'num_items_in_batch'

## Step 7 — Save the LoRA Weights

In [ ]:
LORA_SAVE_DIR = "aria_chopin_lora_final"
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

print(f"✅ LoRA weights saved to '{LORA_SAVE_DIR}/'")
print()
print("Files saved:")
for f in os.listdir(LORA_SAVE_DIR):
    size = os.path.getsize(os.path.join(LORA_SAVE_DIR, f)) / 1e6
    print(f"  {f} ({size:.1f} MB)")
print()
print("💡 These are just the LoRA ADAPTER weights — much smaller than the full model!")
print("   To use them, you load the base model first, then apply these weights on top.")